# Email Agent

Creating an agent that authenticates the user, reads emails from an inbox and drafts a response for HITL review.

#### Design

- Utilize agent state to store the user authentication, create and manage the email state
- Develop a tool that reads the email (use @wrap_model_call to base access on the user type)
- A tool that drafts an email and is interrupted for human review and approval

In [1]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware, ModelRequest, ModelResponse, wrap_model_call
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command, Callable
from langchain.messages import HumanMessage, ToolMessage

from dataclasses import dataclass

from dotenv import load_dotenv

from typing import Dict, Any

In [2]:
load_dotenv()

True

## Setting Up Context and Tools

Next steps:
- Create tools to write passed state objects to the agent graph

In [3]:
class EmailUser(AgentState):
    user_auth: str
    inbox: Dict[str, Any]
    outbox: Dict[str, Any]

In [4]:
@tool
def create_user_inbox(runtime: ToolRuntime,
                      user_auth: str,
                      inbox: Dict[str, Any],
                      outbox: Dict[str, Any]) -> str:
    """Updates the user's inbox, outbox and authorization status.
    Use this tool to update the state when the information is given."""

    try:
        return Command(update={
            "inbox": inbox,
            "outbox": outbox,
            "user_auth": user_auth,
            "messages": [ToolMessage(
                content="Inbox, outbox and user_auth created successfully",
                tool_call_id=runtime.tool_call_id,
        )],
    })
    except Exception as e:
        return f"Could not create inbox, error: {e}"

@tool
def read_inbox(runtime: ToolRuntime) -> str:
    """Reads emails from the user's inbox."""

    try:
        return runtime.state["inbox"]
    except Exception as e:
        return f"Could not read inbox, error: {e}"


@wrap_model_call
def read_inbox_permission(runtime: ToolRuntime,
                          request: ModelRequest,
                          handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Gives permission to the agent to read the user's inbox if they are authorized"""

    if runtime.satate["user_auth"] == "authorized":
        pass
    else:
        tools = None
        request = request.override(tools=tools)

    return handler(request)

@tool
def send_email(runtime: ToolRuntime, body: str) -> Command:
    """Adds a reply to the outbox at the next position."""
    outbox = dict(runtime.state.get("outbox", {}))
    next_position = max((int(k) for k in outbox), default=-1) + 1
    outbox[str(next_position)] = body   # keep keys as strings, consistent with how they're stored
    return Command(update={
        "outbox": outbox,
        "messages": [ToolMessage(content="Email sent successfully", tool_call_id=runtime.tool_call_id)],
    })

## Building the Agent

In [5]:
email_agent = create_agent(
    model="claude-haiku-4-5",
    tools=[create_user_inbox, read_inbox, send_email],
    state_schema=EmailUser,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware[AgentState, None](
            interrupt_on={
                "read_inbox": False,
                "send_email": True
            },
            description_prefix="Tool execution requires approval"
        )
    ],
    system_prompt="""You are a helpful email assistant. You will read my inbox
    and automatically draft responses one at a time, starting with the first emial. 
    Do not ask me for confirmation of the written content, an approval step already
    exists."""
)

In [6]:
user_auth = "authorized"
inbox = {0: {"Hey, can we get some time to talk about the agent project? I am free at 2:00 p.m. CST today."},
         1: {"Please see attached for an update on project X. We are on track to deliver ahead of schedule and below budget!"}}
outbox = {0: {"This is the first message I have sent!"}}

In [7]:
config={"configurable": {"thread_id": "1"}}

response = email_agent.invoke(
    {"messages": [HumanMessage(content=f"{user_auth}, {inbox}, {outbox}")]},
    config=config
)

response

{'messages': [HumanMessage(content="authorized, {0: {'Hey, can we get some time to talk about the agent project? I am free at 2:00 p.m. CST today.'}, 1: {'Please see attached for an update on project X. We are on track to deliver ahead of schedule and below budget!'}}, {0: {'This is the first message I have sent!'}}", additional_kwargs={}, response_metadata={}, id='265fa3d4-af32-42de-b406-8da14b53215d'),
  AIMessage(content=[{'text': "I'll read your inbox and start drafting responses to your emails.", 'type': 'text'}, {'id': 'toolu_01VYRBHP4GavLYeVjVTAC8oG', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_inbox', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CevnwEHZWidKoK3qS62Y2', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_

In [8]:
response = email_agent.invoke(
    Command[tuple[()]](
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config
)

response

{'messages': [HumanMessage(content="authorized, {0: {'Hey, can we get some time to talk about the agent project? I am free at 2:00 p.m. CST today.'}, 1: {'Please see attached for an update on project X. We are on track to deliver ahead of schedule and below budget!'}}, {0: {'This is the first message I have sent!'}}", additional_kwargs={}, response_metadata={}, id='265fa3d4-af32-42de-b406-8da14b53215d'),
  AIMessage(content=[{'text': "I'll read your inbox and start drafting responses to your emails.", 'type': 'text'}, {'id': 'toolu_01VYRBHP4GavLYeVjVTAC8oG', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_inbox', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CevnwEHZWidKoK3qS62Y2', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_